# EDA

Цель анализа — оценить структуру и качество исследуемого датасета, выявить проблемы: некорректные типы данных, пропуски, некорректные значения, дубликаты и аномалии. Данный анализ необходим для дальнейшей обработки данных.

In [ ]:
import pandas as pd

## 1. Структурная проверка данных

In [ ]:
data = pd.read_csv("../data/finance_ecommerce_dirty_dataset.csv")
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.dtypes

### Вывод по разделу 1

Датасет состоит из 6060 строк и 23 столбцов. Часть полей имеет типы, которые не соответствуют их смыслу: `Date` и `CustomerSince` хранятся как строки, `Amount` и `Balance` имеют строковый тип, а `Phone` и `Postal code` определены как числовые значения.

## 2. Целостность и полнота данных

In [ ]:
data.isna().sum()

In [ ]:
data.isna().mean()

Наибольшая доля пропущенных значений наблюдается в `MerchantEmail` (~30%), `MerchantPhone` (~23%) и `Notes` (~13%). В `CardNumber`, `Phone`, `PostalCode` и `Email` доля пропусков составляет ~4–7%, остальные поля заполнены практически полностью.

In [30]:
data.duplicated().sum()

np.int64(1)

В датасете обнаружен 1 полный дубликат строки.

In [31]:
data["TransactionID"].is_unique

False

In [32]:
data["TransactionID"].dropna().is_unique

False

In [35]:
data["TransactionID"].dropna().duplicated().sum()

np.int64(60)

In [ ]:
data.loc[
    data["TransactionID"].notna()
    & data["TransactionID"].duplicated(keep=False),
    "TransactionID"
].value_counts()

`TransactionID` рассматривался как идентификатор транзакции. В столбце есть 3 пропущенных значения, а среди заполненных значений обнаружено 60 дубликатов идентификаторов. Проверка повторяющихся значений показывает, что такие `TransactionID` встречаются по два раза.

### Вывод по разделу 2

Анализ полноты показал, что данные имеют неодинаковую степень заполнености, определены доли пропусков для каждого типа данных. В датаседе обнаружен один полный дубликат строки. Так же был проведен анализ уникальности идентефикатора, который показал, что при дальнейшей обработке необходимо определить порядок работы с его повторяющимися значениями. 

In [ ]:
data["IsFraud"].value_counts(dropna=False)

В `IsFraud` встречаются значения `No`, `Yes`, `Yess` и пропуски

In [ ]:
data["CustomerSince"].value_counts(dropna=False)

В `CustomerSince` используются разные форматы дат, а также встречается значение `unknown`. Поэтому при преобразовании используется `format="mixed"` и `errors="coerce"`.

In [ ]:
data["Amount"].value_counts(dropna=False)

In [ ]:
amount_numeric = pd.to_numeric(data["Amount"], errors="coerce")

In [ ]:
data.loc[amount_numeric.isna(), "Amount"].value_counts(dropna=False)

In [ ]:
balance_numeric = pd.to_numeric(data["Balance"], errors="coerce")

In [ ]:
data.loc[balance_numeric.isna(), "Balance"].value_counts(dropna=False)

В `Amount` и `Balance` встречаются значения, которые нельзя напрямую преобразовать в числовой тип. Поэтому при приведении типов используется `pd.to_numeric(..., errors="coerce")`: некорректные значения становятся пропусками.

In [ ]:
date_converted = pd.to_datetime(data["Date"], errors="coerce", format="mixed")

In [ ]:
data.loc[date_converted.isna(), "Date"].value_counts(dropna=False)

In [ ]:
customer_since_converted = pd.to_datetime(data["CustomerSince"], errors="coerce", format="mixed")

In [ ]:
data.loc[customer_since_converted.isna(), "CustomerSince"].value_counts(dropna=False)

In [ ]:
data["MerchantPhone"].value_counts(dropna=False).head(15)

In [ ]:
data["PostalCode"].value_counts(dropna=False).head(15)

In [ ]:
data["Phone"].value_counts(dropna=False).head(15)


`MerchantPhone`, `PostalCode` и `Phone` используются как идентификаторы, их логичнее хранить как строки, а не как числовые значения.

In [ ]:
data["Date"].head(20)


Для `Date` значения записаны в формате месяц/день/год, поэтому при преобразовании используется формат `%m/%d/%Y`.